# 01 — Exploratory Data Analysis

Load sample images, inspect class distribution, and visualize real vs fake samples.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data.dataset import collect_images_from_dir

DATA_ROOT = Path("data/processed/train")
IMAGE_SIZE = 224

In [ ]:
def load_split_stats(root: Path) -> pd.DataFrame:
    rows = []
    for split in ["train", "val", "test"]:
        split_dir = Path("data/processed") / split
        if not split_dir.exists():
            continue
        for label_name, label in [("real", 0), ("fake", 1)]:
            class_dir = split_dir / label_name
            if not class_dir.exists():
                continue
            count = sum(1 for _ in class_dir.rglob("*.*"))
            rows.append({"split": split, "class": label_name, "count": count})
    return pd.DataFrame(rows)

stats_df = load_split_stats(DATA_ROOT)
stats_df

In [ ]:
if not stats_df.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=stats_df, x="split", y="count", hue="class", ax=ax)
    ax.set_title("Real vs Fake Distribution by Split")
    ax.set_ylabel("Image Count")
    plt.tight_layout()
    plt.show()
else:
    print("No processed data found. Add images under data/processed/{train,val,test}/{real,fake}/")

In [ ]:
def show_samples(root: Path, n_per_class: int = 4) -> None:
    if not root.exists():
        print(f"Directory not found: {root}")
        return
    fig, axes = plt.subplots(2, n_per_class, figsize=(3 * n_per_class, 6))
    for row, (class_name, label) in enumerate([("real", 0), ("fake", 1)]):
        class_dir = root / class_name
        paths = list(class_dir.rglob("*.jpg")) + list(class_dir.rglob("*.png"))
        for col in range(n_per_class):
            ax = axes[row, col]
            if col < len(paths):
                img = cv2.cvtColor(cv2.imread(str(paths[col])), cv2.COLOR_BGR2RGB)
                ax.imshow(img)
                ax.set_title(f"{class_name} #{col}")
            ax.axis("off")
    plt.suptitle("Sample Frames")
    plt.tight_layout()
    plt.show()

show_samples(DATA_ROOT)

In [ ]:
def compute_basic_stats(root: Path) -> pd.DataFrame:
    rows = []
    if not root.exists():
        return pd.DataFrame()
    paths, labels = collect_images_from_dir(root)
    for path, label in zip(paths[:200], labels[:200]):
        img = cv2.imread(path)
        if img is None:
            continue
        h, w = img.shape[:2]
        rows.append({
            "class": "fake" if label == 1 else "real",
            "height": h,
            "width": w,
            "mean_pixel": float(img.mean()),
        })
    return pd.DataFrame(rows)

basic_stats = compute_basic_stats(DATA_ROOT)
if not basic_stats.empty:
    display(basic_stats.groupby("class").agg({"height": "mean", "width": "mean", "mean_pixel": "mean"}))
else:
    print("No images available for stats.")